In [ ]:
# v3: a bigger model, trained from scratch for the chat, sized to be served on Vercel's free plan.
# Why: played as a chat, v2 (12M params) barely answers what it is told. Shakespeare's real reply is only ~2%
# more likely after the line it answered than after a random line, and fine-tuning v2 did not move that.
# Listening takes a bigger model, and a longer memory takes a longer context.
# Size: 8 layers x 512 wide, 27.8M params (111 MB of float32), 1024 tokens of context. The server runs the model
# in numpy on Vercel's Hobby plan: 1 vCPU, 2 GB of memory, 300 s per request. Timed with the server's own code
# on one thread, a chat turn at full context takes ~3.4 s (v2 at 256 tokens: 0.7 s), and 8 at once ~43 s,
# inside the server's 60 s timeout. 10 x 512 was ~10% slower, 12 x 640 almost twice as slow.
# Data: TinyStories (simple English, as v2 learned first), then Shakespeare, ending with dialogue in the chat
# page's format. Tokenizer: v2's 4096 word pieces, so the server needs no new tokenizer.
# This notebook only trains and uploads: it prepares the data, trains, saves v3.npz and uploads it.
# On Kaggle: Settings -> Accelerator: GPU T4 x2, Internet: on. Add-ons -> Secrets: HF_TOKEN (a Hugging Face write
# token) for the upload at the end. Then Save Version -> Save & Run All: it runs in the background (up to 12 hours)
# and v3.npz, ckpt.pt and losses.csv land in the Output tab.
# Imports, seeds, and device: CUDA on Kaggle, MPS on Apple Silicon, otherwise CPU.
import contextlib
import json
import math
import random
import re
import time
import urllib.request
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from tokenizers import Tokenizer
from torch.nn import functional as F

torch.manual_seed(1337)
random.seed(1337)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

In [ ]:
# Where things go. The working folder (/kaggle/working on Kaggle) is the run's output: the checkpoint and the
# trained model end up there. Downloads and prepared data go to /tmp on Kaggle, so they aren't saved with it.
ON_KAGGLE = Path("/kaggle").exists()
DATA = Path("/tmp/tinygpt-data") if ON_KAGGLE else Path("data")
DATA.mkdir(parents=True, exist_ok=True)


def cached(name, make):
    """DATA/<name>, made by make(path) the first time."""
    path = DATA / name
    if not path.exists():
        make(path)
    return path


print("output:", Path(".").resolve(), "| data:", DATA.resolve())

In [ ]:
# The tokenizer: v2's, from its file on the Hugging Face Hub (pinned to a commit), which holds the weights,
# the config and the tokenizer.
V2_URL = "https://huggingface.co/hr5ht/tinygpt-shakespeare-v2/resolve/1840ceb7a940ea483a895831b125dc8afc5ef791/stories.npz"
v2_path = cached("stories.npz", lambda path: urllib.request.urlretrieve(V2_URL, path))
with np.load(v2_path) as f:
    tokenizer_json = str(f["tokenizer"])
tok = Tokenizer.from_str(tokenizer_json)
EOT = tok.token_to_id("<|endoftext|>")


def encode(s):
    return tok.encode(s).ids


def decode(ids):
    return tok.decode(ids)


print(f"{tok.get_vocab_size()} word pieces")

In [ ]:
# The complete works of Shakespeare (Project Gutenberg #100), prepared as in model.ipynb: only the text
# between Gutenberg's START/END markers, curly quotes turned into straight ones. 90/10 train/val split.
URL = "https://www.gutenberg.org/cache/epub/100/pg100.txt"
path = cached("shakespeare.txt", lambda path: urllib.request.urlretrieve(URL, path))
raw = path.read_text(encoding="utf-8-sig")
start = raw.index("\n", raw.index("*** START OF")) + 1
end = raw.index("*** END OF")
text = raw[start:end].replace("\r\n", "\n").strip()
text = text.translate(str.maketrans({"’": "'", "‘": "'", "“": '"', "”": '"'}))

n = int(0.9 * len(text))
shakespeare = {"train": np.array(encode(text[:n]), dtype=np.uint16), "val": np.array(encode(text[n:]), dtype=np.uint16)}
print(f"{len(text):,} characters -> {sum(map(len, shakespeare.values())):,} tokens")

In [ ]:
# TinyStories (roneneldan/TinyStories on the Hub): short stories in simple English, the plain language the
# model learns first. The train file is 1.9 GB (~490M tokens); the run below uses ~180M, so only the first
# TS_TRAIN_TOKENS are tokenized (a few minutes). Stories are separated by <|endoftext|>, the same token
# v2 stops at. Tokens are stored as uint16 and read through a memmap.
TS_URL = "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories-{}.txt"
TS_TRAIN_TOKENS = 200_000_000


def tokenize_stories(split, max_tokens):
    def make(out):
        src = DATA / f"TinyStories-{split}.txt"
        if not src.exists():
            urllib.request.urlretrieve(TS_URL.format(split), src)
        n, rest = 0, ""
        with open(src, encoding="utf-8") as f, open(out, "wb") as w:
            while n < max_tokens:
                chunk = f.read(16 << 20)
                parts = (rest + chunk).split("<|endoftext|>")
                rest = parts.pop() if chunk else ""  # the last part may be a story cut in two
                stories = [s.strip() for s in parts if s.strip()]
                if stories:
                    ids = np.concatenate([np.array(e.ids + [EOT], dtype=np.uint16) for e in tok.encode_batch(stories)])
                    ids.tofile(w)
                    n += len(ids)
                if not chunk:
                    break
        src.unlink()  # the text is no longer needed

    return np.memmap(cached(f"tinystories_{split}.bin", make), dtype=np.uint16, mode="r")


t0 = time.time()
stories = {"train": tokenize_stories("train", TS_TRAIN_TOKENS), "val": tokenize_stories("valid", 10**9)}
print(f"TinyStories: {len(stories['train']):,} train / {len(stories['val']):,} val tokens ({time.time() - t0:.0f}s)")
print(decode(stories["val"][:60].tolist()))

In [ ]:
# Split the plays into scenes, and each scene into (speaker, speech). A speech starts at a line like "ROMEO."
# and runs until the next speaker. Stage directions (indented lines, [_Aside._]) are dropped, and verse that
# resumes after a stage direction stays with its speaker. Headings (ACT, SCENE, play titles) end a scene.
# The last 10% of scenes are held out, which is roughly the last 10% of the text, as in model.ipynb.
SPEAKER = re.compile(r"[A-Z][A-Z' -]*[A-Z]\.")  # "ROMEO.", "LADY MACBETH.", "FIRST CITIZEN."


def clean(lines):
    lines = (re.sub(r"\[[^\]]*\]?", "", line).replace("_", "").strip() for line in lines if not line.startswith(" "))
    return "\n".join(line for line in lines if line)


def split_scenes(text):
    scenes, scene, speech = [], [], None
    for block in text.split("\n\n"):
        lines = block.split("\n")
        if lines[0].startswith((" ", "[")):  # stage direction
            continue
        if SPEAKER.fullmatch(lines[0]) or lines[0].isupper() or lines[0].startswith(("SCENE", "Scene", "ACT", "Act")):
            if speech and speech[1]:
                scene.append(tuple(speech))
            speech = None
            if SPEAKER.fullmatch(lines[0]):
                speech = [lines[0][:-1], clean(lines[1:])]
                continue
            if len(scene) >= 2:  # a heading: the scene so far is done
                scenes.append(scene)
            scene = []
        elif speech is not None:  # verse resuming after a stage direction
            speech[1] = (speech[1] + "\n" + clean(lines)).strip()
    if speech and speech[1]:
        scene.append(tuple(speech))
    if len(scene) >= 2:
        scenes.append(scene)
    return scenes


scenes = split_scenes(text)
n_val = len(scenes) // 10
dialogue = {"train": scenes[:-n_val], "val": scenes[-n_val:]}
print(f"{len(scenes)} scenes, {sum(map(len, scenes)):,} speeches; {n_val} scenes held out")

In [ ]:
# GPT: token + position embeddings -> n_layer pre-norm transformer blocks (causal multi-head self-attention +
# MLP, each with a residual) -> LayerNorm -> output head tied to the token embedding. The same parameter names
# as model.ipynb and the server's numpy port. Attention goes through F.scaled_dot_product_attention, which
# never builds the (T, T) matrix in memory; with a KV cache it feeds one token at a time.
@dataclass
class GPTConfig:
    vocab_size: int
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384
    dropout: float = 0.2


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.n_head, self.dropout = cfg.n_head, cfg.dropout
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
        self.resid_drop = nn.Dropout(cfg.dropout)

    def forward(self, x, cache=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        # (B, T, C) -> (B, n_head, T, head_size)
        q, k, v = (t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v))
        if cache is not None:
            # one new token at a time: it may see every cached position, so it needs no mask
            assert T == 1
            k = torch.cat([cache[0], k], dim=2)
            v = torch.cat([cache[1], v], dim=2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=cache is None, dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(y)), (k, v)


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(cfg.n_embd, 4 * cfg.n_embd),
            nn.GELU(),
            nn.Linear(4 * cfg.n_embd, cfg.n_embd),
            nn.Dropout(cfg.dropout),
        )

    def forward(self, x, cache=None):
        a, cache = self.attn(self.ln1(x), cache)
        x = x + a
        x = x + self.mlp(self.ln2(x))
        return x, cache


class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList(Block(cfg) for _ in range(cfg.n_layer))
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # tied embeddings
        self.apply(self._init_weights)
        # GPT-2: the layers that write into the residual stream start smaller, scaled by depth
        for name, p in self.named_parameters():
            if name.endswith(("attn.proj.weight", "mlp.2.weight")):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, caches=None):
        B, T = idx.shape
        past = 0 if caches is None else caches[0][0].size(2)
        assert past + T <= self.cfg.block_size, "sequence longer than block_size"
        pos = torch.arange(past, past + T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        new_caches = []
        for i, block in enumerate(self.blocks):
            x, c = block(x, None if caches is None else caches[i])
            new_caches.append(c)
        return self.head(self.ln_f(x)), new_caches

In [ ]:
# Dialogue examples, in the chat page's format: "NAME.\nspeech\n\n" per speech, with the scene so far as context.
# - The speaker of the window's last speech plays "the model". The loss covers each of their speeches that
#   answers someone else, including the blank line that ends it, and nothing else. Their "NAME.\n" headers
#   are context too; the chat page writes them.
# - Context is cut from the front at a speech boundary to fit. 30% of windows keep only 1-3 earlier
#   speeches, like the first turns of a chat. Half the time the line being answered is put on one line,
#   the way the chat page sends what you type.
# - The string always runs on into the next speaker's name (not trained on): a blank line before a name is
#   two "\n" tokens, but at the very end of a string it would be one "\n\n" token, which a chat never shows.
def dialogue_example(scenes, block_size, rng=random):
    """(token ids, loss mask over the targets ids[1:])."""
    while True:
        scene = rng.choice(scenes)
        end = rng.randrange(1, len(scene))
        me = scene[end][0]
        if scene[end - 1][0] != me:
            break
    start = max(0, end - rng.randint(1, 3)) if rng.random() < 0.3 else 0
    flatten = rng.random() < 0.5
    after = scene[end + 1][0] if end + 1 < len(scene) else scene[end - 1][0]
    # skip ahead to the speeches that can fit (~3 characters per token here), then drop more until they do
    chars = 0
    for j in range(end, start - 1, -1):
        chars += len(scene[j][0]) + len(scene[j][1]) + 4
        if chars > 3.5 * block_size:
            start = j + 1
            break
    while start < end:
        pieces, spans, at = [], [], 0
        for j in range(start, end + 1):
            name, speech = scene[j]
            if flatten and j == end - 1:
                speech = " ".join(speech.split())
            head, body = f"{name}.\n", f"{speech}\n\n"
            if name == me and j > start and scene[j - 1][0] != me:
                spans.append((at + len(head), at + len(head) + len(body)))
            pieces.append(head + body)
            at += len(head) + len(body)
        enc = tok.encode("".join(pieces) + f"{after}.\n")
        if len(enc.ids) <= block_size + 1:
            break
        start += 1
    else:
        return dialogue_example(scenes, block_size, rng)  # the answer alone doesn't fit: pick another
    # a token is a target if it starts inside an answer; mask[t] is about predicting ids[t + 1]
    mask = [any(a <= s < b for a, b in spans) for s, _ in enc.offsets[1:]]
    return enc.ids, mask


# A batch mixes three kinds of row: a window of TinyStories, a window of plain Shakespeare (loss on every
# token), or a dialogue example (loss on the answers). Rows are padded to the longest; padding has no loss.
def get_batch(split, batch_size, mix, block_size):
    kinds = random.choices(list(mix), weights=list(mix.values()), k=batch_size)
    rows = []
    for kind in kinds:
        if kind == "dialogue":
            rows.append(dialogue_example(dialogue[split], block_size))
            continue
        data = stories[split] if kind == "stories" else shakespeare[split]
        i = random.randrange(len(data) - block_size - 1)
        rows.append((data[i:i + block_size + 1].tolist(), [True] * block_size))
    T = max(len(ids) for ids, _ in rows) - 1
    x = torch.full((batch_size, T), EOT)
    y = torch.full((batch_size, T), EOT)
    mask = torch.zeros((batch_size, T), dtype=torch.bool)
    for r, (ids, m) in enumerate(rows):
        x[r, :len(ids) - 1] = torch.tensor(ids[:-1])
        y[r, :len(ids) - 1] = torch.tensor(ids[1:])
        mask[r, :len(m)] = torch.tensor(m)
    return x.to(device), y.to(device), mask.to(device)

In [ ]:
# The model and the run. 32 x 1024 tokens per step, 16 rows at a time per GPU (a T4 has 15 GB): on Kaggle's
# two T4s that is one micro-batch split across both, on one GPU two micro-batches. 6000 steps is ~200M tokens. AdamW with weight decay on the weight matrices only; linear warmup, then cosine
# decay. Dropout 0.1: the run goes over Shakespeare's 1.7M training tokens a dozen times or so.
# Data mix: the first 85% of steps are 95% TinyStories and 5% Shakespeare. The last 15%, while the learning
# rate winds down, are half TinyStories, a quarter plain Shakespeare and a quarter dialogue.
# fp16 autocast + gradient scaling on CUDA only. The first eval line prints the measured speed and the
# time left; expect a few hours. Lower max_iters for a shorter run.
BLOCK_SIZE = 1024
cfg = GPTConfig(vocab_size=tok.get_vocab_size(), block_size=BLOCK_SIZE, n_layer=8, n_head=8, n_embd=512, dropout=0.1)

n_gpus = max(1, torch.cuda.device_count())
micro_batch, grad_accum = 16 * n_gpus, max(1, 2 // n_gpus)
max_iters = 6000
lr, min_lr = 6e-4, 6e-5
warmup_iters = 300
weight_decay = 0.1
grad_clip = 1.0
eval_interval, eval_iters = 250, 10
ckpt_interval = 500
final_phase = 0.85


def mix_at(it):
    if it < final_phase * max_iters:
        return {"stories": 0.95, "shakespeare": 0.05}
    return {"stories": 0.5, "shakespeare": 0.25, "dialogue": 0.25}


def get_lr(it):
    if it < warmup_iters:
        return lr * (it + 1) / warmup_iters
    progress = (it - warmup_iters) / max(1, max_iters - warmup_iters)
    return min_lr + 0.5 * (lr - min_lr) * (1 + math.cos(math.pi * progress))


use_amp = device == "cuda"
amp = lambda: torch.autocast("cuda", dtype=torch.float16) if use_amp else contextlib.nullcontext()


# The loss of a batch, computed where the batch is: with two GPUs, nn.DataParallel gives each half of the rows
# and gets back two numbers (loss sum, token count) instead of the logits. autocast is entered inside forward
# because DataParallel runs each GPU's half on its own thread.
class LossOf(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x, y, mask):
        with amp():
            logits = self.model(x)[0]
        ce = F.cross_entropy(logits.float().flatten(0, 1), y.flatten(), reduction="none")
        return (ce * mask.flatten()).sum()[None], mask.sum()[None]


model = GPT(cfg).to(device)
loss_of = nn.DataParallel(LossOf(model)) if n_gpus > 1 else LossOf(model)


def batch_loss(x, y, mask):
    total, count = loss_of(x, y, mask)
    return total.sum() / count.sum()


# validation loss per kind of text, with dropout off
@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for kind in ("stories", "shakespeare", "dialogue"):
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            losses[k] = batch_loss(*get_batch("val", micro_batch, {kind: 1}, BLOCK_SIZE)).item()
        out[kind] = losses.mean().item()
    model.train()
    return out


print(f"{n_gpus} GPU(s): {micro_batch} rows per micro-batch, {grad_accum} micro-batch(es) per step")
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M params, {micro_batch * grad_accum * BLOCK_SIZE * max_iters / 1e6:.0f}M tokens planned")

In [ ]:
# Training loop: AdamW, gradient clipping, validation loss every eval_interval steps, a checkpoint (model,
# optimizer, step, loss log, random state) every ckpt_interval steps, written to the output. Training resumes
# from a checkpoint if there is one: in the working folder (rerunning this session), or in an input. To continue
# a run that stopped (Kaggle's 12-hour limit, an error), Add Input -> this notebook's previous version, then
# Save & Run All again.
decay = [p for p in model.parameters() if p.dim() >= 2]
no_decay = [p for p in model.parameters() if p.dim() < 2]
optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": weight_decay}, {"params": no_decay, "weight_decay": 0.0}],
    lr=lr, betas=(0.9, 0.95), **({"fused": True} if device == "cuda" else {}),
)
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
ckpt_path = Path("ckpt.pt")


def save_checkpoint(it, log):
    state = {
        "model": model.state_dict(), "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
        "it": it, "log": log, "config": asdict(cfg), "random": random.getstate(), "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if device == "cuda" else None,
    }
    torch.save(state, ckpt_path.with_suffix(".tmp"))
    ckpt_path.with_suffix(".tmp").replace(ckpt_path)  # a disconnect mid-save never leaves half a checkpoint


start_it, log = 0, []
previous = sorted(Path("/kaggle/input").glob("**/ckpt.pt")) if ON_KAGGLE else []
resume_from = ckpt_path if ckpt_path.exists() else (previous[0] if previous else None)
if resume_from:
    ck = torch.load(resume_from, map_location=device, weights_only=False)
    assert ck["config"] == asdict(cfg), f"{resume_from} is a checkpoint for another config"
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    scaler.load_state_dict(ck["scaler"])
    random.setstate(ck["random"])
    torch.set_rng_state(ck["torch"])
    if ck["cuda"] is not None:
        torch.cuda.set_rng_state_all(ck["cuda"])
    start_it, log = ck["it"], ck["log"]
    print(f"resuming at step {start_it} from {resume_from}")

model.train()
t0, t_last, it_last = time.time(), time.time(), start_it
for it in range(start_it, max_iters):
    for group in optimizer.param_groups:
        group["lr"] = get_lr(it)
    if it % eval_interval == 0 or it == max_iters - 1:
        val = estimate_loss()
        log.append((it, val["stories"], val["shakespeare"], val["dialogue"]))
        speed = (time.time() - t_last) / max(1, it - it_last)
        left = f" | {speed:.2f}s/step, ~{speed * (max_iters - it) / 3600:.1f}h left" if it > it_last else ""
        print(f"step {it:5d} | val: stories {val['stories']:.3f} shakespeare {val['shakespeare']:.3f} "
              f"answers {val['dialogue']:.3f} | {(time.time() - t0) / 60:.0f} min{left}")
        t_last, it_last = time.time(), it

    mix = mix_at(it)
    for _ in range(grad_accum):
        loss = batch_loss(*get_batch("train", micro_batch, mix, BLOCK_SIZE)) / grad_accum
        scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    if (it + 1) % ckpt_interval == 0:
        save_checkpoint(it + 1, log)

save_checkpoint(max_iters, log)
model.eval()
print(f"done; this session trained for {(time.time() - t0) / 60:.0f} min")

In [ ]:
# Save in the inference server's format: the npz holds the weights (head.weight is tok_emb.weight, so it is
# left out), the config and the tokenizer. Plus the torch checkpoint for later fine-tunes, and the loss log.
weights = {k: v.detach().float().cpu().numpy() for k, v in model.state_dict().items() if k != "head.weight"}
np.savez("v3.npz", **weights, config=np.array(json.dumps(asdict(cfg))), tokenizer=np.array(tokenizer_json))
torch.save({"model": model.state_dict(), "config": asdict(cfg)}, "v3.pt")
with open("losses.csv", "w") as f:
    f.write("step,val_stories,val_shakespeare,val_answers\n")
    f.writelines(f"{s},{a:.4f},{b:.4f},{c:.4f}\n" for s, a, b, c in log)
print(f"v3.npz: {Path('v3.npz').stat().st_size / 1e6:.0f} MB, in {Path('.').resolve()}")

In [ ]:
# Upload to the Hugging Face Hub as a new model repo, next to v1 and v2. The server downloads v3.npz.
# The token comes from the Kaggle secret HF_TOKEN (asked for when not on Kaggle), so it is never stored in the
# notebook. Without it, download v3.npz from the Output tab and upload it by hand.
from huggingface_hub import HfApi

token = None
try:
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:  # not on Kaggle, or no such secret
    if not ON_KAGGLE:
        import getpass

        token = getpass.getpass("HF write token: ")
if token:
    api = HfApi(token=token)
    repo_id = f"{api.whoami()['name']}/tinygpt-shakespeare-v3"
    api.create_repo(repo_id, exist_ok=True)
    for f in ("v3.npz", "v3.pt", "losses.csv"):
        api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id=repo_id)
    print(f"uploaded to https://huggingface.co/{repo_id}")
else:
    print("no HF_TOKEN secret: v3.npz is in the Output tab; add the secret and run this cell to upload")